<a href="https://colab.research.google.com/github/MatchLab-Imperial/deep-learning-course/blob/master/08_RL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reinforcement Learning: Planning and Tabular Control

In this tutorial, we enter the world of reinforcement learning (RL) with [FrozenLake](https://gymnasium.farama.org/environments/toy_text/frozen_lake/). We first model the environment as a Markov Decision Process (MDP) and solve it when its transition model is known. We then remove that assumption and learn directly from experience with tabular Q-learning.

This progression separates two important ideas:

- **Planning:** compute a policy from a known model of the environment.
- **Model-free control:** learn action values from sampled transitions without knowing the model.

Part 2 continues from the Q-table to a neural approximation of $Q(s,a)$ with a Deep Q-Network (DQN), then introduces Actor–Critic methods.

In [ ]:
# Enter your CID below before running the submission cells.
CID = "00000000"

## Setup

Install Gymnasium and import the libraries used throughout the notebook.

In [ ]:
from pathlib import Path

SUBMISSION_DIR = Path("./submission")
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

def submission_path(suffix):
    """Return a CID-prefixed path inside the submission folder."""
    student_cid = CID.strip()
    if not student_cid or student_cid == "YOUR_CID":
        raise ValueError("Enter your CID in the CID cell before saving coursework.")
    return SUBMISSION_DIR / f"{student_cid}_{suffix}"


def save_current_cell_code(suffix, start_marker=None, end_marker=None):
    """Save the student-authored part of the currently executing notebook cell."""
    shell = get_ipython()
    source = ""

    if shell is not None and hasattr(shell, "get_parent"):
        try:
            source = shell.get_parent().get("content", {}).get("code", "")
        except Exception:
            source = ""

    if not source and shell is not None:
        history = getattr(getattr(shell, "history_manager", None),
                          "input_hist_raw", [])
        if history:
            source = history[-1]

    if not source:
        raise RuntimeError("Could not read the current notebook cell for submission.")

    if start_marker is not None:
        start = source.find(start_marker)
        if start < 0:
            raise ValueError(f"Start marker not found: {start_marker}")
        source = source[start:]

    if end_marker is not None:
        end = source.find(end_marker)
        if end >= 0:
            source = source[:end]

    output_path = submission_path(suffix)
    output_path.write_text(source.rstrip() + "\n", encoding="utf-8")
    print(f"Saved: {output_path}")
    return output_path


print(f"Coursework files will be saved in: {SUBMISSION_DIR.resolve()}")


In [ ]:
!pip install gymnasium --upgrade

import base64
import collections
import glob
import io
import os
import random
import time
import torch

from IPython import display as ipythondisplay
from IPython.display import HTML
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

def set_seed(seed: int) -> None:
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Achieving the Goal with a Markov Decision Process

Before asking an agent to learn, suppose that we already know how the environment works. A **Markov Decision Process (MDP)** describes a sequential decision problem with the tuple

$$\mathcal{M}=(\mathcal{S},\mathcal{A},P,R,\gamma),$$

where $\mathcal{S}$ is the set of states, $\mathcal{A}$ is the set of actions, $P(s'\mid s,a)$ is the probability of reaching $s'$ after taking action $a$ in $s$, $R(s,a,s')$ is the reward, and $\gamma$ discounts future rewards. The Markov property says that the next-state distribution depends on the current state and action, not on the complete history.

Frozen Lake is a small finite MDP:

- each of the 16 grid cells is a state;
- the actions are left, down, right, and up;
- slippery ice makes transitions stochastic;
- reaching `G` gives reward 1, while every other transition gives reward 0;
- entering a hole or the goal ends the episode.

Because Gymnasium exposes the transition model for this teaching environment, we can **plan** with it instead of learning it from sampled experience.

In [ ]:
mdp_env = gym.make("FrozenLake-v1", is_slippery=True)
transition_model = mdp_env.unwrapped.P

print("Lake map:")
print(mdp_env.unwrapped.desc.astype(str))
print("\nPossible outcomes from the start after choosing LEFT:")
for probability, next_state, reward, terminated in transition_model[0][0]:
    print(f"p={probability:.3f}, next_state={next_state}, "
          f"reward={reward}, terminal={terminated}")

## Planning with value iteration

The state value $V^*(s)$ is the best expected discounted return obtainable from state $s$. Value iteration repeatedly applies the Bellman optimality backup

$$V_{k+1}(s)=\max_a\sum_{s'}P(s'\mid s,a)\left[R(s,a,s')+\gamma V_k(s')\right].$$

Once the values stop changing, choosing the action that maximises the same expression gives an optimal policy $\pi^*(s)$. This policy maximises the **probability of eventually reaching the goal** here because the only positive reward is received at `G`.

In [ ]:
def value_iteration(env, gamma=0.99, tolerance=1e-10):
    """Return optimal state values and a greedy policy for a finite MDP."""
    model = env.unwrapped.P
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    values = np.zeros(n_states)

    while True:
        old_values = values.copy()
        for state in range(n_states):
            action_values = []
            for action in range(n_actions):
                expected_return = sum(
                    probability * (reward + gamma * old_values[next_state] * (not terminated))
                    for probability, next_state, reward, terminated
                    in model[state][action]
                )
                action_values.append(expected_return)
            values[state] = max(action_values)

        if np.max(np.abs(values - old_values)) < tolerance:
            break

    policy = np.zeros(n_states, dtype=int)
    for state in range(n_states):
        action_values = np.zeros(n_actions)
        for action in range(n_actions):
            action_values[action] = sum(
                probability * (reward + gamma * values[next_state] * (not terminated))
                for probability, next_state, reward, terminated
                in model[state][action]
            )
        policy[state] = np.argmax(action_values)

    return values, policy


mdp_values, mdp_policy = value_iteration(mdp_env)
print("Optimal values:\n", np.round(mdp_values.reshape(4, 4), 3))

## Follow and evaluate the planned policy

The arrows below show the action selected in every safe state. A single rollout may still fail because the lake is slippery, so we estimate goal achievement over many independent episodes.

In [ ]:
action_symbols = np.array(["←", "↓", "→", "↑"])
lake = mdp_env.unwrapped.desc.astype(str)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(np.zeros((4, 4)), cmap="Blues", vmin=0, vmax=1)
for row in range(4):
    for col in range(4):
        tile = lake[row, col]
        label = action_symbols[mdp_policy[row * 4 + col]] if tile in {"S", "F"} else tile
        ax.text(col, row, label, ha="center", va="center", fontsize=22)
ax.set_xticks(np.arange(-0.5, 4, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 4, 1), minor=True)
ax.grid(which="minor", color="white", linewidth=2)
ax.tick_params(which="both", bottom=False, left=False,
               labelbottom=False, labelleft=False)
ax.set_title("Policy obtained from the known MDP")
plt.show()

episodes = 1000
successes = 0
for episode in range(episodes):
    state, _ = mdp_env.reset(seed=episode)
    for _ in range(100):
        state, reward, terminated, truncated, _ = mdp_env.step(mdp_policy[state])
        if terminated or truncated:
            successes += int(reward == 1)
            break

print(f"Reached the goal in {successes}/{episodes} episodes "
      f"(success rate: {successes / episodes:.1%}).")
mdp_env.close()

Value iteration works because the full transition and reward model is available. In many real problems it is unknown or too large to enumerate. **Q-learning is model-free:** it estimates useful action values directly from transitions sampled while the agent interacts with the environment.

# Q-Learning

This family of RL methods try to learn an approximator of the action-value functions $Q(s,a)$  based on the [Bellman equation](https://en.wikipedia.org/wiki/Bellman_equation), such that the update using a classical [gradient descent ](https://en.wikipedia.org/wiki/Gradient_descent) formulation is given by:
$$Q\left(s,a\right)=Q\left(s,a\right)+ \alpha \left(r+\gamma \max _{a} Q\left(s_{t+1},a\right)-Q\left(s,a\right)\right).$$
Where $\alpha$ is the step size.
 Q-Learning updates the estimated reward at each time step and  uses the old estimate $ \max _{a}Q\left(s_{t+1},a\right)$ to update the new ones. In a more algorithmic way, the Q-Learning process is the following:


1.   Initialize Q-values at random $Q\left(s,a\right)$.
2. Forever or until learning is stopped do:
> 1.  Observe state $s$.
> 2.   Take action $a$ according to your policy, e.g., $\epsilon$-greedy.
> 3.   Observe reward $r$ and new state $s_{t+1}$.
> 4. Based on your actual estimates, compute $\max _{a}Q\left(s_{t+1},a\right)$.
> 5. Update your current estimate for  $Q\left(s,a\right)$:
$$Q\left(s,a\right)=Q\left(s,a\right)+ \alpha \left(r+\gamma \max _{a} Q\left(s_{t+1},a\right)-Q\left(s,a\right)\right).$$

Okay, now that we are familiar with Q-Learning lets jump to a real implementation of it.







## Tabular Q-Learning with Frozen Lake
In this section we will teach an agent how to play  the [Frozen lake](https://gym.openai.com/envs/FrozenLake-v0/) game using a classical tabular Q-learning. Brace yourselves, winter is coming!

![alt text](https://raw.githubusercontent.com/simoninithomas/Deep_reinforcement_learning_Course/1ee37cfc3130057f828f19b3cee6066d41c1eeb4/Q%20learning/FrozenLake/frozenlake.png)

Winter has arrived and you and your friends were tossing around a frisbee at the park when you made a wild throw that left the frisbee out in the middle of the lake. The water is mostly frozen, but there are a few holes where the ice has melted. If you step into one of those holes, you'll fall into the freezing water. At this time, there's an international frisbee shortage, so you must navigate across the lake and retrieve the disc. However, the ice is slippery, so you won't always move in the direction you intend.
The goal of this game is to go from the starting state (S) to the goal state (G) by walking only on frozen tiles (F) and avoid holes (H). However, the ice is slippery (!!), so you won't always move in the direction you intend (stochastic environment), i.e., there is a probability $p$ that you move in the direction selected and a probability $(1-p)$ that given the slippery ice, you move to a random position near position. Specifically, let's say you select the action UP, you have a probability of 1/3 of actually going UP, 1/3 of going RIGHT and 1/3 of going LEFT. Similarly, if you select LEFT, you have a probability of 1/3 of actually going LEFT, 1/3 of going UP and 1/3 of going DOWN.

The lake is represented by a 4x4 grid and the location where the frisbee has landed (G) as well as the holes (H) is always the same for every new game. The game is restarted every time you have successfully recovered the frisbee or you have fallen into the cold waters. A reward of +1 is given every time you recover the frisbee and 0 other way.


**Environment creation:**

OpenAi is  a library composed of many environments that we can use to train our agents, in our case we choose to use the Frozen Lake.

In [ ]:
env = gym.make("FrozenLake-v1", render_mode='rgb_array')

**Q-table**

 Now, we'll create our Q-table. The goal of the Q-table is to store the estimates $Q\left(s,a\right)$ and retrieve them when necessary. In this game the states are represented by each of the 16 grid positions being 0 the starting position and 16 the goal position and the actions are 4: left, right, up and down. Our Q-table will have then $16 \times 4$ positions, where the value of the first column of the first row represents the expected return of being in position 0 and taking left.

The number of rows (states) and columns (actions) the table will have can also be obtained using the values action_size and the state_size from the OpenAI Gym library: *env.action_space.n* and* env.observation_space.n*.

We initialize the table to 0.

In [ ]:
action_size = env.action_space.n
state_size = env.observation_space.n
qtable = np.zeros((state_size, action_size))
print(qtable)

**Hyperparameters**

Following, we specify the hyperparameters:


In [ ]:
total_episodes = 25000        # Total episodes
learning_rate = 0.8           # Learning rate (alpha in the previous formulation)
max_steps = 100               # Max steps per episode
gamma = 0.95                  # Discounting rate

At first, we don't know how to interact with the environment (Q-table values set to 0), so we start exploring it by taking a random action with probability $\epsilon=1$, capturing the rewards obtained and updating the Q-values of the table accordingly. As time passes by, we start knowing more and more the environment, so we reduce (decay_rate) the probability of taking a random action and we start exploiting our knowledge, we choose the action that leads us to the highest reward, i.e., the one with the highest Q-value.

In [ ]:
# Exploration parameters
max_epsilon = 1.0             # Exploration probability at start
min_epsilon = 0.01            # Minimum exploration probability
epsilon = max_epsilon         # Exploration rate
decay_rate = 0.001            # Exponential decay rate for exploration prob

**Q-Learning**

Now we implement the Q-Learning algorithm:
> 1.  Observe state $s$.
> 2.   Choose a random value $v$ between 0 and 1.
> 3. If $v<\epsilon$, we choose a random action, otherwise we select the action with maximum $Q(s,a)$.
> 3.   Observe reward $r$ and new state $s_{t+1}$.
> 4. Based on your previous estimates, compute $\max _{a}Q\left(s_{t+1},a\right)$.
> 5. Update your current estimates for  $Q\left(s,a\right)$:
$$Q\left(s,a\right)=Q\left(s,a\right)+ \alpha \left(r+\gamma \max _{a} Q\left(s_{t+1},a\right)-Q\left(s,a\right)\right).$$


In [ ]:
set_seed(0)

# List of rewards
rewards = []

for episode in range(total_episodes):
    # Reset the environment
    state, _ = env.reset()
    step = 0
    done = False
    total_rewards = 0

    for step in range(max_steps):
        # 3. Choose an action a in the current world state (s)
        ## First we randomize a number
        exp_exp_tradeoff = random.uniform(0, 1)

        ## If this number > greater than epsilon --> exploitation (taking the biggest Q value for this state)
        if exp_exp_tradeoff > epsilon:
            action = np.argmax(qtable[state,:])

        # Else doing a random choice --> exploration
        else:
            action = env.action_space.sample()

        # Take the action (a) and observe the outcome state(s') and reward (r)
        new_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        # Update Q(s,a):= Q(s,a) + lr [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
        # qtable[new_state,:] : all the actions we can take from new state
        qtable[state, action] = qtable[state, action] + learning_rate * (reward + gamma * np.max(qtable[new_state, :]) - qtable[state, action])

        total_rewards += reward

        # Our new state is state
        state = new_state

        # If done (if we're dead) : finish episode
        if done == True:
            break

    # Reduce epsilon (because we need less and less exploration)
    epsilon = min_epsilon + (max_epsilon - min_epsilon)*np.exp(-decay_rate*episode)
    rewards.append(total_rewards)

print ("Score over time: " +  str(sum(rewards)/total_episodes))
print(qtable)

**Use our Q-table to play FrozenLake!**

After 25000 episodes, our Q-table can be used as a "cheatsheet" to play FrozenLake"!
  
By running this cell, you can see our agent playing FrozenLake:

In [ ]:
env = gym.make("FrozenLake-v1", render_mode='rgb_array')
state, _ = env.reset()
step = 0

plt.imshow(env.render())
plt.show()

for step in range(max_steps):

    # Take the action (index) that have the maximum expected future reward given that state
    action = np.argmax(qtable[state,:])

    new_state, reward, terminated, truncated, info = env.step(action)
    plt.imshow(env.render())
    plt.show()

    # We print the current step.
    print(f"Number of steps: {step}")
    if terminated or truncated:
      break
    state = new_state

env.close()

Let’s see how many times our agent finds the frisbee 🎉

To this end we will print the last step of the game.

In [ ]:
set_seed(0)

games=5
for game in range(games):
    env = gym.make("FrozenLake-v1")
    state, _ = env.reset()
    step = 0
    for step in range(max_steps):

        # Take the action (index) that have the maximum expected future reward given that state
        action = np.argmax(qtable[state,:])
        new_state, reward, terminated, truncated, info = env.step(action)

        if terminated or truncated:
        # Here, we decide to only print the last state (to see if our agent is on the goal or fall into a hole)
        # We print the number of step it took.
            print(f"Succeed: {reward == 1}, Number of steps: {step}")
            break
        state = new_state
    env.close()

In [ ]:
set_seed(0)

games = 1000
total_rewards = 0

for game in range(games):
    env = gym.make("FrozenLake-v1")
    state, _ = env.reset()
    step = 0
    for step in range(max_steps):
        # Take the action (index) that have the maximum expected future reward given that state
        action = np.argmax(qtable[state,:])
        new_state, reward, terminated, truncated, info = env.step(action)
        if terminated or truncated:
            total_rewards += reward
            break
        state = new_state
    env.close()
success = total_rewards / games
print("Ratio of sucessfully finished episodes is {:f}".format(success))

# Coursework


## Task 1 — Reward design for Q-learning on FrozenLake

FrozenLake's default reward is **sparse**: the agent receives `1` only when it reaches the goal and `0` for every other transition. Most early episodes therefore provide little information about which choices were useful.

In this task you will train a tabular Q-learning agent, design a new reward definition, and test whether it improves learning. The environment's real objective does not change: performance is always measured by the percentage of episodes that reach `G`. Your designed reward is used only as the learning signal.

### Learning objectives

1. Implement reward shaping without changing the Q-learning algorithm.
2. Compare reward definitions in a controlled experiment.
3. Explain why a shaped reward helps, fails to help, or creates an unintended behaviour.


### 1. Experiment configuration

The lake remains slippery, so an intended action may move the agent in a different direction. The constants use an `FL_` prefix so this task can be run in the same notebook as Task 2 without overwriting its settings.


In [ ]:
FL_ENV_NAME = "FrozenLake-v1"
FL_IS_SLIPPERY = True
FL_EPISODES = 15_000
FL_MAX_STEPS = 100
FL_ALPHA = 0.20
FL_GAMMA = 0.99
FL_EPSILON_START = 1.0
FL_EPSILON_END = 0.05
FL_EPSILON_DECAY = 0.0005
FL_TRAIN_SEED = 21

fl_env = gym.make(FL_ENV_NAME, is_slippery=FL_IS_SLIPPERY)
FL_LAKE = fl_env.unwrapped.desc.astype(str)
FL_TILES = FL_LAKE.reshape(-1)
FL_GOAL_STATE = int(np.flatnonzero(FL_TILES == "G")[0])
FL_GOAL_ROW, FL_GOAL_COL = divmod(FL_GOAL_STATE, FL_LAKE.shape[1])

print(FL_LAKE)
print(f"States: {fl_env.observation_space.n}; actions: {fl_env.action_space.n}")
fl_env.close()


### 2. Q-learning trainer

The trainer below is complete. It accepts a reward function as an argument, which lets us change the learning signal without changing any other part of the experiment. The recorded `successes` always use Gymnasium's original reward, not the designed reward.


In [ ]:
def fl_choose_action(q_values, epsilon, rng):
    """Choose an epsilon-greedy action with random tie-breaking."""
    if rng.random() < epsilon:
        return int(rng.integers(len(q_values)))
    best_actions = np.flatnonzero(q_values == np.max(q_values))
    return int(rng.choice(best_actions))


def sparse_reward(state, action, next_state, env_reward,
                  terminated, truncated, step):
    """FrozenLake's original reward definition."""
    return float(env_reward)


def train_frozen_lake(reward_function, seed=FL_TRAIN_SEED):
    """Train tabular Q-learning using the supplied learning reward."""
    env = gym.make(FL_ENV_NAME, is_slippery=FL_IS_SLIPPERY)
    rng = np.random.default_rng(seed)
    q_table = np.zeros((env.observation_space.n, env.action_space.n),
                       dtype=np.float64)
    successes, shaped_returns = [], []

    for episode in range(FL_EPISODES):
        state, _ = env.reset(seed=seed + episode)
        epsilon = FL_EPSILON_END + (FL_EPSILON_START - FL_EPSILON_END) * np.exp(
            -FL_EPSILON_DECAY * episode
        )
        episode_success = 0
        shaped_return = 0.0

        for step in range(FL_MAX_STEPS):
            action = fl_choose_action(q_table[state], epsilon, rng)
            next_state, env_reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            learning_reward = reward_function(
                state, action, next_state, env_reward,
                terminated, truncated, step
            )
            next_value = 0.0 if done else np.max(q_table[next_state])
            td_target = learning_reward + FL_GAMMA * next_value
            q_table[state, action] += FL_ALPHA * (
                td_target - q_table[state, action]
            )

            shaped_return += learning_reward
            episode_success = max(episode_success, int(env_reward == 1))
            state = next_state
            if done:
                break

        successes.append(episode_success)
        shaped_returns.append(shaped_return)

    env.close()
    return {"q_table": q_table,
            "successes": np.asarray(successes),
            "shaped_returns": np.asarray(shaped_returns)}


def evaluate_frozen_lake(q_table, episodes=2_000, seed=50_000):
    """Evaluate the greedy policy using the real goal-success objective."""
    env = gym.make(FL_ENV_NAME, is_slippery=FL_IS_SLIPPERY)
    rng = np.random.default_rng(seed)
    successes = 0

    for episode in range(episodes):
        state, _ = env.reset(seed=seed + episode)
        for _ in range(FL_MAX_STEPS):
            action = fl_choose_action(q_table[state], epsilon=0.0, rng=rng)
            state, env_reward, terminated, truncated, _ = env.step(action)
            if terminated or truncated:
                successes += int(env_reward == 1)
                break

    env.close()
    return successes / episodes


### 3. Train the sparse-reward baseline

Run the unchanged FrozenLake reward first. This is the baseline your design must be compared against.


In [ ]:
fl_sparse_result = train_frozen_lake(sparse_reward)
fl_sparse_success = evaluate_frozen_lake(fl_sparse_result["q_table"])
print(f"Sparse-reward greedy success rate: {fl_sparse_success:.1%}"

### 4. Your reward definition — edit only the next cell

Change `reward` in the function below. Everything useful for a reward design is exposed locally:

- `tile` is `"F"`, `"H"`, or `"G"` for the tile just entered;
- `state` and `next_state` are integer positions, and their row/column coordinates are provided;
- `old_distance` and `new_distance` are Manhattan distances to the goal;
- `action`, `step`, `terminated`, and `truncated` are also available.

Possible ingredients include a goal bonus, a hole penalty, a small cost per step, or a progress signal. These are prompts, not requirements: define and justify your own reward. Keep the function deterministic and return one finite number. **Do not edit the Q-learning trainer.**


In [ ]:
def designed_reward(state, action, next_state, env_reward,
                    terminated, truncated, step):
    """Return your self-designed learning reward for one transition."""
    tile = FL_TILES[next_state]
    row, col = divmod(state, FL_LAKE.shape[1])
    next_row, next_col = divmod(next_state, FL_LAKE.shape[1])
    old_distance = abs(row - FL_GOAL_ROW) + abs(col - FL_GOAL_COL)
    new_distance = abs(next_row - FL_GOAL_ROW) + abs(next_col - FL_GOAL_COL)

    # TODO: Replace this default with your own reward definition.
    # Examples of available conditions:
    # if tile == "H": ...
    # if tile == "G": ...
    # if new_distance < old_distance: ...
    reward = float(env_reward)

    return float(reward)


# Auto-save the completed reward function when this cell runs.
save_current_cell_code(
    "E8_T1_Reward_Function_Implementation.txt",
    start_marker="def designed_reward",
    end_marker="# Auto-save",
)


### 5. Train and compare the two reward definitions

This cell uses the same seeds and hyperparameters for both runs. The learning curves show the fraction of episodes that reached the real goal in a 250-episode window. The final numbers come from 2,000 new greedy-policy episodes.


In [ ]:
fl_designed_result = train_frozen_lake(designed_reward)
fl_designed_success = evaluate_frozen_lake(fl_designed_result["q_table"])

FL_WINDOW = 250
def fl_moving_success(successes, window=FL_WINDOW):
    return np.convolve(successes, np.ones(window) / window, mode="valid")

fig, ax = plt.subplots(figsize=(9, 4.5))
for label, result in [("Sparse reward", fl_sparse_result),
                      ("Designed reward", fl_designed_result)]:
    curve = fl_moving_success(result["successes"])
    episodes = np.arange(len(curve)) + FL_WINDOW
    ax.plot(episodes, curve, label=label)
ax.set_xlabel("Training episode")
ax.set_ylabel(f"Goal rate (last {FL_WINDOW} episodes)")
ax.set_title("FrozenLake Q-learning: effect of the learning reward")
ax.set_ylim(-0.02, 1.02)
ax.grid(alpha=0.25)
ax.legend()

evaluation_text = (
    f"Greedy evaluation over 2,000 episodes\n"
    f"Sparse: {fl_sparse_success:.1%}   "
    f"Designed: {fl_designed_success:.1%}   "
    f"Change: {fl_designed_success - fl_sparse_success:+.1%}"
)
ax.text(0.5, -0.20, evaluation_text, transform=ax.transAxes,
        ha="center", va="top", fontsize=9)
caption = (
    "Caption (replace this placeholder with 2–3 sentences): "
    "Briefly explain your understanding of the graph and its main findings."
)
fig.text(0.5, 0.02, caption, ha="center", va="bottom", wrap=True, fontsize=9)
fig.subplots_adjust(bottom=0.38)

output_path = submission_path("E8_T1_Learning_Curve.png")
fig.savefig(output_path, dpi=200, bbox_inches="tight")
print(f"Saved: {output_path}")
plt.show()

print(f"Sparse reward evaluation:   {fl_sparse_success:.1%}")
print(f"Designed reward evaluation: {fl_designed_success:.1%}")
print(f"Absolute change:             {fl_designed_success - fl_sparse_success:+.1%}")


### Task 1 submission

Submit the following to Canvas:

After entering your CID in the CID cell below the title, running the relevant code cells automatically saves the code and figure in `./submission`:

1. Your completed `designed_reward` function.

   File name: `CID_E8_T1_Reward_Function_Implementation.txt`

2. The learning-curve figure with the evaluation rates. Replace the placeholder caption with **2–3 sentences** that briefly explain your understanding of the graph and its main findings.

   File name: `CID_E8_T1_Learning_Curve.png`

3. A short interpretation describing your reward, why you expected it to help, whether it improved the real goal-success rate, and any unintended incentive it might create.

   File name: `CID_E8_T1_Reward_Interpretation.txt`

---


## Task 2 — On-policy and off-policy learning with CliffWalking

In **CliffWalking**, the agent starts at the bottom-left of a 4×12 grid and must reach the bottom-right goal. The cells between them are a cliff. Each ordinary transition receives reward `-1`. Stepping into the cliff receives `-100` and returns the agent to the start; reaching the goal ends the episode.

![Animated CliffWalking Q-learning rollout](assets/cliffwalking-qlearning.gif)

*Animated Q-learning rollout. Source: [VachanVY/Reinforcement-Learning](https://github.com/VachanVY/Reinforcement-Learning).*

The environment has 48 discrete states and four actions, so a Q-table is sufficient. The shortest route runs immediately above the cliff. It is efficient if every action is greedy, but a single exploratory downward action can cause a large penalty.

### Learning objectives

1. Implement Q-learning as an off-policy method and SARSA as an on-policy method.
2. Distinguish the greedy target policy from the exploring behaviour policy.
3. Use learned routes and state-visitation patterns to explain why SARSA can learn a safer policy.


### 1. Experiment configuration

The main comparison uses the classical deterministic CliffWalking setting: $\gamma=1$, a fixed $\epsilon=0.1$, and `is_slippery=False`. Keeping exploration active is important because SARSA learns the value of the behaviour policy it is actually following. Use the same settings and seed for both algorithms.


In [ ]:
ENV_NAME = "CliffWalking-v1"
EPISODES = 500
MAX_STEPS = 200
ALPHA = 0.5
GAMMA = 1.0
EPSILON = 0.1
IS_SLIPPERY = False
TRAIN_SEED = 7

env = gym.make(ENV_NAME, is_slippery=IS_SLIPPERY)
print(f"States: {env.observation_space.n}; actions: {env.action_space.n}; "
      f"slippery: {IS_SLIPPERY}")
env.close()


### 2. The shared behaviour policy

Both agents collect experience with the same **$\epsilon$-greedy behaviour policy**:

- with probability $1-\epsilon$, choose an action with the largest current Q-value;
- with probability $\epsilon$, choose a random action.

Here $\epsilon$ remains at `0.1` during training and behaviour-policy evaluation. Random tie-breaking avoids always preferring the first action when several Q-values are equal. The helper is complete and should not be modified.


In [ ]:
def choose_action(q_values, epsilon, rng):
    """Sample an epsilon-greedy action with random tie-breaking."""
    if rng.random() < epsilon:
        return int(rng.integers(len(q_values)))
    best = np.flatnonzero(q_values == np.max(q_values))
    return int(rng.choice(best))


### 3. The learning target

Both algorithms update the observed state-action pair $(s,a)$ after receiving $(r,s')$. They differ only in the value used at the next state.

**Q-learning (off-policy)** learns about a greedy target policy even while the behaviour policy explores:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\left[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\right].$$

**SARSA (on-policy)** learns about the same $\epsilon$-greedy policy that generates the experience. It samples the next behaviour action $a'$ and uses that action in its target:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\left[r + \gamma Q(s',a') - Q(s,a)\right].$$

Near the cliff, Q-learning's target assumes that the next action will be greedy. SARSA's target includes the consequences of occasional exploratory actions. A route with more space below it can therefore have a better value for SARSA, even though it takes more steps.


### 4. Train Q-learning first

Q-learning is provided as a complete baseline. Its target always uses the largest action value at the next state, while its next behaviour action is sampled separately with the $\epsilon$-greedy policy.

Run this cell first. It should train without any student changes.


In [ ]:
def train_q_learning(seed=TRAIN_SEED, episodes=EPISODES,
                     alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON):
    env = gym.make(ENV_NAME, is_slippery=IS_SLIPPERY)
    rng = np.random.default_rng(seed)
    q_table = np.zeros((env.observation_space.n, env.action_space.n), dtype=np.float64)
    rewards, steps, cliff_falls, td_errors = [], [], [], []

    for episode in range(episodes):
        state, _ = env.reset(seed=seed + episode)
        total_reward, fell, episode_td_errors = 0, False, []

        for step in range(MAX_STEPS):
            action = choose_action(q_table[state], epsilon, rng)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward
            fell = fell or (reward == -100)

            # Off-policy target: learn about the greedy next action.
            next_value = 0.0 if done else np.max(q_table[next_state])
            target = reward + gamma * next_value
            td_error = target - q_table[state, action]
            q_table[state, action] += alpha * td_error
            episode_td_errors.append(abs(td_error))

            if done:
                break
            state = next_state

        rewards.append(total_reward)
        steps.append(step + 1)
        cliff_falls.append(int(fell))
        td_errors.append(np.mean(episode_td_errors))

    env.close()
    return {"q_table": q_table, "rewards": np.array(rewards),
            "steps": np.array(steps), "cliff_falls": np.array(cliff_falls),
            "td_errors": np.array(td_errors)}


q_learning_result = train_q_learning()
print("Q-learning training complete")


### 5. Complete and train SARSA

SARSA is shown separately so that its action flow is easy to compare with Q-learning. Unlike Q-learning, SARSA selects the next behaviour action **before** updating and uses that same action both in the target and in the next environment step.

Complete only the two marked parts. Use these hints:

1. For a non-terminal transition, select `next_action` with `choose_action`, then use `q_table[next_state, next_action]` as `next_value`. For a terminal transition, use zero and set `next_action` to `None`.
2. After a non-terminal update, carry the **same** `next_action` into the next iteration.

Then run the cell to train SARSA.


In [ ]:
def train_sarsa(seed=TRAIN_SEED, episodes=EPISODES,
                alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON):
    env = gym.make(ENV_NAME, is_slippery=IS_SLIPPERY)
    rng = np.random.default_rng(seed)
    q_table = np.zeros((env.observation_space.n, env.action_space.n), dtype=np.float64)
    rewards, steps, cliff_falls, td_errors = [], [], [], []

    for episode in range(episodes):
        state, _ = env.reset(seed=seed + episode)
        action = choose_action(q_table[state], epsilon, rng)
        total_reward, fell, episode_td_errors = 0, False, []

        for step in range(MAX_STEPS):
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward
            fell = fell or (reward == -100)

            # TODO 1: Construct the SARSA bootstrap value.
            # Hint: SARSA learns about the action its behaviour policy will
            # actually take next. A terminal state has no future value.
            raise NotImplementedError("Complete the SARSA next-action target.")

            # On-policy target: use the next action selected by the behaviour policy.
            target = reward + gamma * next_value
            td_error = target - q_table[state, action]
            q_table[state, action] += alpha * td_error
            episode_td_errors.append(abs(td_error))

            if done:
                break

            # TODO 2: Advance the state-action pair.
            # Hint: reuse the action involved in the bootstrap target rather
            # than asking the behaviour policy for another sample.
            raise NotImplementedError("Carry the SARSA state and action forward.")

        rewards.append(total_reward)
        steps.append(step + 1)
        cliff_falls.append(int(fell))
        td_errors.append(np.mean(episode_td_errors))

    env.close()
    return {"q_table": q_table, "rewards": np.array(rewards),
            "steps": np.array(steps), "cliff_falls": np.array(cliff_falls),
            "td_errors": np.array(td_errors)}


sarsa_result = train_sarsa()
print("SARSA training complete")

# Auto-save the completed SARSA implementation when this cell runs.
save_current_cell_code(
    "E8_T2_SARSA_Implementation.txt",
    start_marker="def train_sarsa",
    end_marker="# Auto-save",
)


### 6. Collect the two results

Both agents have now been trained independently with the same seed, environment and $\epsilon$-greedy behaviour policy. Store them together for the controlled comparison below.


In [ ]:
results = {
    "Q-learning": q_learning_result,
    "SARSA": sarsa_result,
}


### 7. Compare training returns

Each curve is the mean return over a 20-episode moving window. Returns include exploratory actions and cliff penalties, so they measure how well each agent behaves while it is still using $\epsilon=0.1$. Because returns are negative, a curve closer to zero is better.


In [ ]:
def moving_average(values, window=20):
    return np.convolve(values, np.ones(window) / window, mode="valid")

fig, ax = plt.subplots(figsize=(9, 4.5))
for name, result in results.items():
    smooth = moving_average(result["rewards"])
    ax.plot(np.arange(len(smooth)) + 20, smooth, label=name)
ax.set_xlabel("Training episode")
ax.set_ylabel("Mean return (20-episode moving average)")
ax.set_title("CliffWalking training with fixed $\epsilon=0.1$")
ax.legend()
ax.grid(alpha=0.25)
caption = (
    "Caption (replace this placeholder with 2–3 sentences): "
    "Briefly explain your understanding of the graph and its main findings."
)
fig.text(0.5, 0.02, caption, ha="center", va="bottom", wrap=True, fontsize=9)
fig.subplots_adjust(bottom=0.22)

output_path = submission_path("E8_T2_Return_vs_Episode.png")
fig.savefig(output_path, dpi=200, bbox_inches="tight")
print(f"Saved: {output_path}")
plt.show()


### 8. Evaluate both the greedy and behaviour policies

Two evaluations answer different questions:

- **Greedy (`epsilon=0`)** shows the route preferred by the final Q-table when there is no exploration.
- **Behaviour (`epsilon=0.1`)** measures the policy SARSA was trained to improve and exposes the risk of acting near the cliff.

`mean_cliff_distance` is the mean vertical distance from visited non-terminal states to the cliff row. Larger values indicate that the agent usually travels farther from the cliff.


In [ ]:
def evaluate_policy(q_table, epsilon, episodes=1000, seed=10_000):
    env = gym.make(ENV_NAME, is_slippery=IS_SLIPPERY)
    rng = np.random.default_rng(seed)
    returns, steps, successes, falls, distances = [], [], 0, 0, []
    visits = np.zeros(env.observation_space.n, dtype=np.int64)

    for episode in range(episodes):
        state, _ = env.reset(seed=seed + episode)
        total_reward, fell = 0, False

        for step in range(MAX_STEPS):
            action = choose_action(q_table[state], epsilon, rng)
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            fell = fell or (reward == -100)
            visits[state] += 1

            row, _ = divmod(state, 12)
            if state not in {36, 47}:  # Exclude start and goal from distance summary.
                distances.append(3 - row)
            if terminated or truncated:
                break

        returns.append(total_reward)
        steps.append(step + 1)
        successes += int(terminated and state == 47)
        falls += int(fell)

    env.close()
    return {"mean_return": float(np.mean(returns)),
            "mean_steps": float(np.mean(steps)),
            "success_rate": successes / episodes,
            "cliff_rate": falls / episodes,
            "mean_cliff_distance": float(np.mean(distances)),
            "visits": visits.reshape(4, 12)}

evaluations = {}
for name, result in results.items():
    for label, epsilon in [("greedy", 0.0), ("behaviour", EPSILON)]:
        evaluations[(name, label)] = evaluate_policy(result["q_table"], epsilon)

evaluation_df = __import__("pandas").DataFrame({
    key: {metric: value for metric, value in data.items() if metric != "visits"}
    for key, data in evaluations.items()
}).T
evaluation_df.index.names = ["algorithm", "evaluation_policy"]
rounded_evaluation = evaluation_df.round(3)
ipythondisplay.display(rounded_evaluation)

# Render the executed table as a submission-ready image.
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.axis("off")
row_labels = [f"{algorithm} / {policy}"
              for algorithm, policy in rounded_evaluation.index]
table = ax.table(cellText=rounded_evaluation.values,
                 colLabels=rounded_evaluation.columns,
                 rowLabels=row_labels,
                 cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.0, 1.5)
ax.set_title("CliffWalking policy evaluation", pad=18)

output_path = submission_path("E8_T2_Evaluation_Table.png")
fig.savefig(output_path, dpi=200, bbox_inches="tight")
print(f"Saved: {output_path}")
plt.show()


### 9. Visualise the learned routes using State-Visitation Heatmaps

The black line is the final greedy route. The blue background shows average state visits per episode over 1,000 rollouts while the agent continues to use $\epsilon=0.1$. Red cells are the cliff, `S` is the start, and `G` is the goal.

This figure makes the central comparison visible: Q-learning normally chooses the short route one row above the cliff, while SARSA tends to move into the upper rows to reduce the cost of exploratory mistakes. The behaviour-policy heatmap shows that this is not just one selected episode; it summarises all evaluation rollouts.


In [ ]:
def greedy_path(q_table):
    env = gym.make(ENV_NAME, is_slippery=IS_SLIPPERY)
    state, _ = env.reset(seed=0)
    path = [state]
    for _ in range(MAX_STEPS):
        best = np.flatnonzero(q_table[state] == np.max(q_table[state]))
        action = int(best[0])
        state, _, terminated, truncated, _ = env.step(action)
        path.append(state)
        if terminated or truncated:
            break
    env.close()
    return path

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), constrained_layout=True)
for ax, (name, result) in zip(axes, results.items()):
    visits = evaluations[(name, "behaviour")]["visits"] / 1000
    image = ax.imshow(np.log1p(visits), cmap="Blues", origin="upper")

    for col in range(1, 11):
        ax.add_patch(plt.Rectangle((col - 0.5, 2.5), 1, 1,
                                   facecolor="firebrick", alpha=0.8))

    path = greedy_path(result["q_table"])
    rows, cols = zip(*(divmod(state, 12) for state in path))
    ax.plot(cols, rows, "o-", color="black", linewidth=2, markersize=4)
    label_box = {"facecolor": "white", "alpha": 0.8, "edgecolor": "none", "pad": 1}
    ax.text(0, 3, "S", ha="center", va="center", color="black",
            fontweight="bold", bbox=label_box)
    ax.text(11, 3, "G", ha="center", va="center", color="black",
            fontweight="bold", bbox=label_box)
    ax.set_xticks(range(12))
    ax.set_yticks(range(4))
    ax.set_xlim(-0.5, 11.5)
    ax.set_ylim(3.5, -0.5)
    ax.grid(color="white", linewidth=0.8)
    ax.set_title(f"{name}: greedy route ({len(path) - 1} steps)", fontsize=11)

fig.colorbar(image, ax=axes, shrink=0.75, label="log(1 + mean visits per episode)")
caption = (
    "Caption (replace this placeholder with 2–3 sentences): "
    "Briefly explain your understanding of the graph and its main findings."
)
fig.text(0.5, -0.03, caption, ha="center", va="top", wrap=True, fontsize=9)
output_path = submission_path("E8_T2_State_Visitation_Heatmap.png")
fig.savefig(output_path, dpi=200, bbox_inches="tight")
print(f"Saved: {output_path}")
plt.show()


## Task 2 submission

After entering your CID in the CID cell below the title, running the relevant code cells automatically saves the code and three figures in `./submission`. You should submit the following to Canvas:

1. Your completed **two-part SARSA implementation**.

   File name: `CID_E8_T2_SARSA_Implementation.txt`

2. A **Return vs Episode** figure for Q-learning and SARSA.

   File name: `CID_E8_T2_Return_vs_Episode.png`

3. The **Executed Evaluation table** containing both greedy and behaviour-policy rows.

   File name: `CID_E8_T2_Evaluation_Table.png`

4. The **State-Visitation Heatmap** for both algorithms.

   File name: `CID_E8_T2_State_Visitation_Heatmap.png`

For both required graphs (Return vs Episode and State-Visitation Heatmap), replace the placeholder caption with **2–3 sentences** that briefly explain your understanding and key findings, including the most important difference between Q-learning and SARSA.

5. A short **interpretation** that answers all three questions:
   - Which algorithm learned the shorter greedy route, and which learned the route farther from the cliff? Refer to the figure and `mean_cliff_distance`.
   - Why can Q-learning have a safe greedy evaluation but a much worse return or cliff rate when $\epsilon=0.1$?
   - How does the different bootstrap target cause SARSA to account for exploratory actions while Q-learning does not?

   File name: `CID_E8_T2_SARSA_Q_Learning_Interpretation.txt`
